# Phase 9 - Integrated Python visualisation

This notebook combines the validated Phase 5 terrain and LiDAR display sample, Phase 7 mixed-LoD buildings, and Phase 8 public utility records. Utility depth is unknown: the dashed utility layer is draped 0.75 m above terrain for visibility only and is not an underground position.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from uqgems.integrated_scene import (
    SceneDisplayConfig,
    build_plotly_figure,
    load_integrated_scene_data,
)
from uqgems.phase9 import run_phase9

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

summary = run_phase9(PROJECT_ROOT)
print(f"Status: {summary['status']}")
print(f"Cache reused: {summary['cache_reused']}")
print(f"Checks: {sum(summary['checks'].values())}/{len(summary['checks'])}")
print(summary["decision_gate"])

## Scene layers and controls

Source coordinates remain authoritative. The renderer uses reversible local XY coordinates, and the utility terrain drape exists only in the display layer.

In [ ]:
layers = pd.read_csv(PROJECT_ROOT / "reports/tables/phase9_layer_inventory.csv")
controls = pd.read_csv(PROJECT_ROOT / "reports/tables/phase9_controls.csv")
display(layers)
display(controls)

## Interactive scene

Use the legend to toggle layers and the menus for camera, extent and terrain opacity. Hover over a building or public utility for its attributes. The LiDAR preview is hidden initially and can be enabled from the legend.

In [ ]:
data = load_integrated_scene_data(PROJECT_ROOT, maximum_point_cloud_points=30_000)
config = SceneDisplayConfig()
figure = build_plotly_figure(data, config)
figure.show()

## Rebuild with another display configuration

Configuration affects display geometry only. For example, use `SceneDisplayConfig(vertical_exaggeration=1.5, terrain_opacity=0.25, utility_colour_by='accuracy')`, or colour by `network_type`, `confidence` or `age`. Age uses the asset survey date, which is unknown for every current record. You can also pass `selected_building_ids` and `selected_asset_ids` tuples. Never interpret the utility drape or its vertical exaggeration as physical depth.

## Fixed visual checks

In [ ]:
for filename in [
    "phase9_integrated_plan.png",
    "phase9_integrated_oblique.png",
    "phase9_utilities_cutaway.png",
    "phase9_cross_section.png",
]:
    display(Image(filename=PROJECT_ROOT / "reports/figures" / filename, width=900))

## Interpretation

The terrain/building workflow is suitable for a public-data pilot demonstration. The utilities are not a complete campus network: only two short boundary-crossing lines and one node were found, all without defensible vertical information. Use the scene to demonstrate the integration method and quantify the data gap when requesting UQ-controlled or asset-owner records.